# 📋 規格書（請直接在這一格修改）

學號：＿＿＿＿＿＿　姓名：＿＿＿＿＿＿

## 1. 系統目的（一句話）
讓民眾能透過自然語言完成診所內科、外科、牙科的掛號、取消掛號與剩餘名額查詢。

## 2. 使用者會怎麼說（至少 5 句）
1. 我想要掛號內科
2. 請幫我取消牙科的掛號
3. 請問外科還有名額嗎？
4. 我肚子有點痛，該掛哪一科？
5. 幫我預約心臟科

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|--------|--------|--------|--------|
|clinics|department|TEXT  | 科別名稱（主鍵）|
|clinics|quota  |INTEGER | 每日總名額   |
|clinics|registered|INTEGER | 已掛號人數   |

初始資料：
- 內科: 總名額 10，已掛號 0
- 外科: 總名額 5，已掛號 0
- 牙科: 總名額 8，已掛號 0

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：`掛號`、`取消`、`查詢`
- 對象只能是：`內科`、`外科`、`牙科`
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：提供任何醫療建議、診斷或根據症狀推薦科別。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 剩餘名額 > 0 | 成功掛號 [科別]！ | 已掛號人數 +1 |
| 掛號 | 剩餘名額 = 0 | 抱歉，[科別]名額已滿。 | 無變化 |
| 取消 | 已掛號人數 > 0 | 已成功為您取消 [科別] 掛號。 | 已掛號人數 -1 |
| 取消 | 已掛號人數 = 0 | 您目前並無 [科別] 的掛號紀錄。 | 無變化 |
| 查詢 | 無條件限制 | [科別] 目前剩餘名額為 X 人。 | 無變化 |

## 6. 例外情況（至少 3 個）
1. 使用者未提及具體科別或僅描述症狀（例如：「我頭痛」）：系統回應「抱歉，無法辨識科別。請直接說明要掛號的科別（內科/外科/牙科）或洽詢櫃台。」
2. 使用者欲掛號的科別不在系統服務範圍內（例如：「掛號心臟科」）：系統回應「本診所僅提供內科、外科、牙科之服務，請重新選擇。」
3. 欲掛號之科別名額已滿：系統回應「抱歉，[科別]名額已滿。」

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛內科 | 掛號,內科 |
| 2 | 正常 | 請幫我取消牙科掛號 | 取消,牙科 |
| 3 | 正常 | 查詢外科還有沒有名額 | 查詢,外科 |
| 4 | 錯誤 | 我肚子很痛 | 未知,未知 |
| 5 | 錯誤 | 請幫我掛號心臟科 | 未知,未知 |
| 6 | 錯誤 | 隨便幫我掛號 | 未知,未知 |
| 7 | 邊界 | 外科預約 | 掛號,外科 |
| 8 | 邊界 | 我想看一下牙齒剩多少名額 | 查詢,牙科 |

In [ ]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('SDDB_Lecture05'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 SDDB_Lecture05 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [ ]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 說明：建立診所名額表 clinics 與病患掛號紀錄表 registrations
# ============================================================
conn = sqlite3.connect("clinic.db")
try:
    cursor = conn.cursor()
    # 清除舊表
    cursor.execute("DROP TABLE IF EXISTS registrations")
    cursor.execute("DROP TABLE IF EXISTS clinics")

    # 1. 科別名額表
    cursor.execute("""
        CREATE TABLE clinics (
            department TEXT PRIMARY KEY,
            quota INTEGER NOT NULL,
            registered INTEGER NOT NULL
        )
    """)

    # 2. 病患掛號紀錄表（用於身份核對）
    cursor.execute("""
        CREATE TABLE registrations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id TEXT NOT NULL,
            department TEXT NOT NULL,
            status TEXT DEFAULT 'active'
        )
    """)

    # 新增初始資料 (內科 10 人, 外科 5 人, 牙科 8 人)
    initial_data = [
        ("內科", 10, 0),
        ("外科", 5, 0),
        ("牙科", 8, 0)
    ]
    cursor.executemany("INSERT INTO clinics VALUES (?, ?, ?)", initial_data)
    conn.commit()
    print("✅ 資料庫 clinics 與 registrations 初始化成功！")
finally:
    conn.close()

✅ 資料庫 clinics 與 registrations 初始化成功！


In [ ]:
# ============================================================
# 第 3 格：顯示資料表
# 說明：定義 show_tables() 函式，用於觀察數據與掛號紀錄
# ============================================================
def show_tables():
    conn = sqlite3.connect("clinic.db")
    try:
        print("【科別名額統計】")
        df_clinics = pd.read_sql_query("SELECT department AS 科別, quota AS 總名額, registered AS 已掛號人數, (quota - registered) AS 剩餘名額 FROM clinics", conn)
        display(df_clinics)

        print("\n【病患有效掛號紀錄表】")
        df_regs = pd.read_sql_query("SELECT id AS 掛號單號, patient_id AS 病患ID, department AS 科別, status AS 狀態 FROM registrations WHERE status = 'active'", conn)
        display(df_regs)
    finally:
        conn.close()

show_tables()

【科別名額統計】


,科別,總名額,已掛號人數,剩餘名額
0,內科,10,0,10
1,外科,5,0,5
2,牙科,8,0,8



【病患有效掛號紀錄表】


,掛號單號,病患ID,科別,狀態


In [ ]:
# ============================================================
# 第 4 格：AI 解讀函式
# 說明：呼叫 Groq API 將使用者輸入解析為 動作,科別，並包含許可清單過濾
# ============================================================
def parse_user_intent(user_text):
    prompt = f"""
你是一個診所掛號系統的意圖解析員。
請閱讀使用者輸入，並將其轉化為格式：`動作,科別`。

【允許的動作】
- 掛號
- 取消
- 查詢

【允許的科別】
- 內科
- 外科
- 牙科

【嚴格規則】
1. 若使用者僅描述症狀（如「我肚子痛」、「頭痛」），絕對不可推薦科別，輸出 `未知,未知`。
2. 若提到的科別不在許可清單內，或無法明確判斷動作與科別，輸出 `未知,未知`。
3. 只能輸出 `動作,科別`，不可包含額外文字或標點。

使用者輸入："{user_text}"
輸出結果：
"""
    raw_response = "未知,未知"
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw_response = response.choices[0].message.content.strip()
    except Exception as e:
        raw_response = "未知,未知"

    # 許可清單過濾（白名單檢查）
    valid_actions = ["掛號", "取消", "查詢"]
    valid_departments = ["內科", "外科", "牙科"]

    parsed_action = "未知"
    parsed_dept = "未知"

    if "," in raw_response:
        parts = raw_response.split(",")
        if len(parts) == 2:
            act, dept = parts[0].strip(), parts[1].strip()
            if act in valid_actions:
                parsed_action = act
            if dept in valid_departments:
                parsed_dept = dept

    return parsed_action, parsed_dept, raw_response

In [ ]:
# ============================================================
# 第 5 格：測試清單
# 說明：執行測試清單，驗證 AI 解讀函式的準確度
# ============================================================
test_cases = [
    ("我要掛內科", "掛號", "內科"),
    ("請幫我取消牙科掛號", "取消", "牙科"),
    ("查詢外科還有沒有名額", "查詢", "外科"),
    ("我肚子很痛", "未知", "未知"),
    ("請幫我掛號心臟科", "未知", "未知"),
    ("隨便幫我掛號", "未知", "未知"),
    ("外科預約", "掛號", "外科"),
    ("我想看一下牙齒剩多少名額", "查詢", "牙科")
]

passed_count = 0
print("====== 開始測試 AI 解讀函式 ======\n")

for idx, (text, exp_act, exp_dept) in enumerate(test_cases, 1):
    act, dept, raw = parse_user_intent(text)
    is_passed = (act == exp_act and dept == exp_dept)
    if is_passed:
        passed_count += 1

    status = "✅ 通過" if is_passed else "❌ 失敗"
    print(f"測試 #{idx} [{status}]")
    print(f"  使用者輸入: {text}")
    print(f"  AI 原始回覆: {raw}")
    print(f"  系統解讀: 動作={act}, 科別={dept} (預期: {exp_act},{exp_dept})\n")

print(f"====== 測試完成：通過 {passed_count}/{len(test_cases)} 句 ======")

====== 開始測試 AI 解讀函式 ======

測試 #1 [✅ 通過]
  使用者輸入: 我要掛內科
  AI 原始回覆: 掛號,內科
  系統解讀: 動作=掛號, 科別=內科 (預期: 掛號,內科)

測試 #2 [✅ 通過]
  使用者輸入: 請幫我取消牙科掛號
  AI 原始回覆: 取消,牙科
  系統解讀: 動作=取消, 科別=牙科 (預期: 取消,牙科)

測試 #3 [✅ 通過]
  使用者輸入: 查詢外科還有沒有名額
  AI 原始回覆: 查詢,外科
  系統解讀: 動作=查詢, 科別=外科 (預期: 查詢,外科)

測試 #4 [✅ 通過]
  使用者輸入: 我肚子很痛
  AI 原始回覆: 未知,未知
  系統解讀: 動作=未知, 科別=未知 (預期: 未知,未知)

測試 #5 [✅ 通過]
  使用者輸入: 請幫我掛號心臟科
  AI 原始回覆: 未知,未知
  系統解讀: 動作=未知, 科別=未知 (預期: 未知,未知)

測試 #6 [✅ 通過]
  使用者輸入: 隨便幫我掛號
  AI 原始回覆: 未知,未知
  系統解讀: 動作=未知, 科別=未知 (預期: 未知,未知)

測試 #7 [✅ 通過]
  使用者輸入: 外科預約
  AI 原始回覆: 掛號,外科
  系統解讀: 動作=掛號, 科別=外科 (預期: 掛號,外科)

測試 #8 [✅ 通過]
  使用者輸入: 我想看一下牙齒剩多少名額
  AI 原始回覆: 查詢,牙科
  系統解讀: 動作=查詢, 科別=牙科 (預期: 查詢,牙科)

====== 測試完成：通過 8/8 句 ======


In [ ]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# 說明：帶入 patient_id，確保取消掛號時只針對該病患自己的紀錄
# ============================================================
def handle_registration(action, department, patient_id):
    if not patient_id or not patient_id.strip():
        return "⚠️ 請輸入有效的身分證字號或手機號碼以進行操作。"

    if action == "未知" or department == "未知":
        return "抱歉，無法明確辨識您的需求或科別。請直接說明要掛號的科別（內科、外科、牙科）或洽詢櫃台。"

    conn = sqlite3.connect("clinic.db")
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT quota, registered FROM clinics WHERE department = ?", (department,))
        row = cursor.fetchone()

        if row is None:
            return f"抱歉，本診所目前未提供 {department} 之服務。"

        quota, registered = row

        if action == "掛號":
            # 檢查該病患是否已經掛過這一科（避免重複掛號）
            cursor.execute(
                "SELECT id FROM registrations WHERE patient_id = ? AND department = ? AND status = 'active'",
                (patient_id, department)
            )
            if cursor.fetchone() is not None:
                return f"⚠️ 病患【{patient_id}】您已經掛號過【{department}】，請勿重複掛號。"

            if registered < quota:
                # 1. 寫入該病患的掛號紀錄
                cursor.execute(
                    "INSERT INTO registrations (patient_id, department, status) VALUES (?, ?, 'active')",
                    (patient_id, department)
                )
                # 2. 更新總掛號人數
                cursor.execute("UPDATE clinics SET registered = registered + 1 WHERE department = ?", (department,))
                conn.commit()
                return f"✅ 成功為病患【{patient_id}】完成【{department}】掛號！"
            else:
                return f"❌ 抱歉，【{department}】今日名額已滿。"

        elif action == "取消":
            # 精準對照：只尋找屬「該病患本人」且「有效」的掛號紀錄
            cursor.execute(
                "SELECT id FROM registrations WHERE patient_id = ? AND department = ? AND status = 'active'",
                (patient_id, department)
            )
            reg_row = cursor.fetchone()

            if reg_row is None:
                return f"❌ 取消失敗：查無病患【{patient_id}】在【{department}】的有效掛號紀錄。"

            reg_id = reg_row[0]
            # 1. 將該病患的紀錄設為已取消
            cursor.execute("UPDATE registrations SET status = 'cancelled' WHERE id = ?", (reg_id,))
            # 2. 扣減名額
            cursor.execute("UPDATE clinics SET registered = registered - 1 WHERE department = ?", (department,))
            conn.commit()
            return f"✅ 已成功為病患【{patient_id}】取消【{department}】的掛號。"

        elif action == "查詢":
            remaining = quota - registered
            return f"ℹ️ 【{department}】目前剩餘名額為 {remaining} 人（總名額 {quota} 人，已掛號 {registered} 人）。"

        else:
            return "無法處理該動作。"

    finally:
        conn.close()

In [ ]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# 說明：模擬不同病患操作，驗證能否成功防止病患修改他人的掛號
# ============================================================
def process_user_request(user_input, patient_id):
    action, dept, raw = parse_user_intent(user_input)
    response = handle_registration(action, dept, patient_id)
    return response

print("====== 權限隔離與業務流程測試 ======\n")

# 模擬測試場景：
# 1. 病患 A (A123) 掛號內科
# 2. 病患 B (B456) 嘗試取消內科（應該要失敗，因為他沒掛）
# 3. 病患 A (A123) 取消內科（應該成功）

steps = [
    ("我要掛內科", "A123", "病患 A 進行內科掛號"),
    ("幫我取消內科掛號", "B456", "病患 B 試圖取消內科掛號（非本人）"),
    ("幫我取消內科掛號", "A123", "病患 A 取消自己的內科掛號（本人）")
]

for req, pid, desc in steps:
    print(f"【情境】{desc}")
    print(f"  病患 ID：{pid} | 輸入：{req}")
    reply = process_user_request(req, pid)
    print(f"  系統回應：{reply}\n")

print("====== 測試後資料庫最新狀態 ======")
show_tables()

====== 權限隔離與業務流程測試 ======

【情境】病患 A 進行內科掛號
  病患 ID：A123 | 輸入：我要掛內科
  系統回應：✅ 成功為病患【A123】完成【內科】掛號！

【情境】病患 B 試圖取消內科掛號（非本人）
  病患 ID：B456 | 輸入：幫我取消內科掛號
  系統回應：❌ 取消失敗：查無病患【B456】在【內科】的有效掛號紀錄。

【情境】病患 A 取消自己的內科掛號（本人）
  病患 ID：A123 | 輸入：幫我取消內科掛號
  系統回應：✅ 已成功為病患【A123】取消【內科】的掛號。

====== 測試後資料庫最新狀態 ======
【科別名額統計】


,科別,總名額,已掛號人數,剩餘名額
0,內科,10,0,10
1,外科,5,0,5
2,牙科,8,0,8



【病患有效掛號紀錄表】


,掛號單號,病患ID,科別,狀態


In [31]:
# ============================================================
# 第 8 格：Gradio 網頁
# 說明：架設 Gradio 互動介面（修正為最新 Chatbot 字典格式，防止格式報錯）
# ============================================================
def clinic_service(patient_id, user_message):
    action, dept, raw = parse_user_intent(user_message)
    reply = handle_registration(action, dept, patient_id)
    return reply

with gr.Blocks(title="🏥 診所智能掛號系統") as demo:
    gr.Markdown("# 🏥 診所智能掛號系統")
    gr.Markdown("請先輸入您的**病患識別碼（如手機號碼或身分證號）**，再輸入想執行的動作。")

    with gr.Row():
        patient_id_input = gr.Textbox(label="病患識別碼 (ID / 手機號碼)", placeholder="例如：0912345678", value="A123456789")

    # 指定 type="messages" 以相容最新的 Gradio 對話格式
    chatbot = gr.Chatbot(label="對話紀錄", type="messages")
    msg_input = gr.Textbox(label="請輸入您的需求", placeholder="例如：我要掛內科、查詢牙科名額、取消內科掛號")
    clear_btn = gr.ClearButton([msg_input, chatbot])

    def respond(patient_id, message, chat_history):
        bot_message = clinic_service(patient_id, message)
        # 使用最新的字典格式 append 訊息
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return "", chat_history

    msg_input.submit(respond, [patient_id_input, msg_input, chatbot], [msg_input, chatbot])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9a0b34c184aeffeefb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9a0b34c184aeffeefb.gradio.live
